# Serve a security LLM on Colab's free GPU → cyberhackmythos

Runs an OpenAI-compatible endpoint (Ollama) on Colab's free **T4 GPU** and exposes it
with a public tunnel. Paste the printed URL + model into cyberhackmythos' **Custom
endpoint** (the model switcher in the command bar).

**Runtime → Change runtime type → T4 GPU** before running.

Default model: `WhiteRabbitNeo` (security-specific, won't over-refuse legitimate
security work). Swap the `MODEL` variable for any Ollama tag. For the original
27B OpenMythos you need an A100 (Colab Pro) or a 4-bit GGUF build.

In [ ]:
# 1) Install + start Ollama (OpenAI-compatible server on :11434)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!nvidia-smi -L

In [ ]:
# 2) Pull the security model (8B fits a free T4). Swap for any Ollama tag.
MODEL = "monotykamary/whiterabbitneo-v1.5a"
!ollama pull {MODEL}

In [ ]:
# 3) Expose it with a public tunnel (cloudflared — no signup required)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import subprocess, re, time
proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    print(line, end="")
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("\n\n==============================================================")
print("Paste these into cyberhackmythos → model switcher → Custom endpoint:")
print("  Base URL :", (url or "<tunnel failed — re-run>") + "/v1")
print("  Model    :", MODEL)
print("  API key  : ollama   (any non-empty value works)")
print("==============================================================")

In [ ]:
# 4) (optional) sanity-check the endpoint speaks OpenAI + tool calls
from openai import OpenAI
c = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
r = c.chat.completions.create(model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=16)
print(r.choices[0].message.content)

Keep this notebook running while you use the endpoint — closing the Colab tab tears
down the tunnel. The `trycloudflare.com` URL is ephemeral; re-run cell 3 for a new one.

In cyberhackmythos: command bar → model dropdown → **Custom endpoint…** → paste the
Base URL, Model, and API key above → **Apply**.